### ML Model for Delsys Data Input (LDA Model)
Consists of:
- Filtering
- Windowing
- Feature Extraction
- Training LDA
- Evaluating accuracy
- Works for ADLs like “water-bottle lift” and “zipper”
- Real Time Feedback Loop

First setting up virtual environment:
- py -m venv venv
- venv\Scripts\activate

Then setting up dependencies:
- pip install numpy scipy scikit-learn matplotlib seaborn

#### Data Preprocessing

In [53]:
# Imports
import pandas as pd
import numpy as np
import os
import glob
import pickle
from scipy.signal import butter, filtfilt, iirnotch
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')

In [54]:
def load_emg_imu(csv_path):
    """
    Load EMG + IMU data from Delsys Trigno CSV with proper sensor identification.
    
    Returns:
        emg_data: dict {sensor_name: (n_samples, 1)}
        imu_data: dict {sensor_name: (n_samples, 6)} - [acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z]
        time_data: dict {sensor_name: (n_samples, 1)}
        fs_emg: EMG sampling frequency
        fs_imu: IMU sampling frequency
    """
    # Read sensor row (row 4)
    sensor_row = pd.read_csv(csv_path, header=None, skiprows=3, nrows=1, skipinitialspace=True)
    sensor_row = sensor_row.ffill(axis=1).iloc[0].tolist()

    # Read measurement row (row 6)
    meas_row = pd.read_csv(csv_path, header=None, skiprows=5, nrows=1, skipinitialspace=True)
    meas_row = meas_row.iloc[0].tolist()

    # Create unique column names
    combined_cols = []
    sensor_counter = {}
    for sensor, meas in zip(sensor_row, meas_row):
        sensor = str(sensor).strip() if pd.notna(sensor) else "Unknown"
        meas = str(meas).strip() if pd.notna(meas) else "Unknown"
        
        sensor_counter[sensor] = sensor_counter.get(sensor, 0) + 1
        unique_sensor = f"{sensor}_{sensor_counter[sensor]}"
        combined_cols.append(f"{unique_sensor}||{meas}")

    # Load data starting from row 9
    df = pd.read_csv(csv_path, header=None, skiprows=8, skipinitialspace=True, 
                     low_memory=False, on_bad_lines='skip')

    # Handle column count mismatch
    n_cols = df.shape[1]
    if len(combined_cols) < n_cols:
        for i in range(n_cols - len(combined_cols)):
            combined_cols.append(f"Extra_{i}||Unknown")
    elif len(combined_cols) > n_cols:
        combined_cols = combined_cols[:n_cols]

    df.columns = combined_cols

    # Convert to numeric
    df = df.apply(lambda x: pd.to_numeric(x.astype(str).str.strip(), errors='coerce'))
    df = df.dropna(how='all')  # Remove completely empty rows

    # Organize data by sensor
    sensor_data = {}
    
    for col in df.columns:
        if '||' not in col:
            continue
            
        sensor_name, meas_name = col.split('||')
        
        if sensor_name not in sensor_data:
            sensor_data[sensor_name] = {
                'emg': None,
                'time_emg': None,
                'acc_x': None, 'acc_y': None, 'acc_z': None,
                'gyro_x': None, 'gyro_y': None, 'gyro_z': None,
                'time_imu': None
            }
        
        data_col = df[col].dropna().values
        
        # Classify measurement type
        if '(mV)' in meas_name and 'EMG' in meas_name:
            sensor_data[sensor_name]['emg'] = data_col
        elif 'Time Series' in meas_name and 'EMG' in meas_name:
            sensor_data[sensor_name]['time_emg'] = data_col
        elif 'ACC X' in meas_name:
            sensor_data[sensor_name]['acc_x'] = data_col
        elif 'ACC Y' in meas_name:
            sensor_data[sensor_name]['acc_y'] = data_col
        elif 'ACC Z' in meas_name:
            sensor_data[sensor_name]['acc_z'] = data_col
        elif 'GYRO X' in meas_name:
            sensor_data[sensor_name]['gyro_x'] = data_col
        elif 'GYRO Y' in meas_name:
            sensor_data[sensor_name]['gyro_y'] = data_col
        elif 'GYRO Z' in meas_name:
            sensor_data[sensor_name]['gyro_z'] = data_col
        elif 'Time Series' in meas_name and 'ACC' in meas_name:
            sensor_data[sensor_name]['time_imu'] = data_col

    # Extract organized data
    emg_data = {}
    imu_data = {}
    time_data = {}
    
    for sensor_name, data in sensor_data.items():
        # EMG data
        if data['emg'] is not None and len(data['emg']) > 0:
            emg_data[sensor_name] = data['emg'].reshape(-1, 1)
            if data['time_emg'] is not None:
                time_data[sensor_name] = data['time_emg'].reshape(-1, 1)
        
        # IMU data (stack acc + gyro)
        imu_channels = []
        for key in ['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z']:
            if data[key] is not None and len(data[key]) > 0:
                imu_channels.append(data[key])
        
        if len(imu_channels) > 0:
            # Find minimum length to align all IMU channels
            min_len = min(len(ch) for ch in imu_channels)
            imu_channels = [ch[:min_len] for ch in imu_channels]
            imu_data[sensor_name] = np.column_stack(imu_channels)

    # Determine sampling frequencies from data
    fs_emg = 963  # Hz (from Delsys spec)
    fs_imu = 148.148  # Hz (from Delsys spec)
    
    return emg_data, imu_data, time_data, fs_emg, fs_imu

In [ ]:
def extract_emg_features(window, fs=963):
    """Extract comprehensive EMG features from a window"""
    # Time-domain features
    mav = np.mean(np.abs(window))
    rms = np.sqrt(np.mean(window**2))
    var = np.var(window)
    wl = np.sum(np.abs(np.diff(window)))
    
    # Zero crossings
    zc = np.sum(np.diff(np.sign(window)) != 0) / len(window)
    
    # Slope sign changes
    diff_signal = np.diff(window)
    ssc = np.sum(np.diff(np.sign(diff_signal)) != 0) / (len(window) - 1)
    
    # Frequency-domain features
    fft_vals = np.fft.rfft(window)
    power_spectrum = np.abs(fft_vals)**2
    freqs = np.fft.rfftfreq(len(window), 1/fs)
    
    total_power = np.sum(power_spectrum)
    if total_power > 0:
        mnf = np.sum(freqs * power_spectrum) / total_power
        cumsum = np.cumsum(power_spectrum)
        mdf_idx = np.argmax(cumsum >= total_power/2)
        mdf = freqs[mdf_idx]
    else:
        mnf = 0
        mdf = 0
    
    return np.array([mav, rms, var, wl, zc, ssc, mnf, mdf])

In [56]:
def extract_imu_features(window):
    """Extract IMU features from a window (works for any number of axes)"""
    # Time-domain features
    mean_val = np.mean(window, axis=0)
    std_val = np.std(window, axis=0)
    rms_val = np.sqrt(np.mean(window**2, axis=0))
    range_val = np.max(window, axis=0) - np.min(window, axis=0)
    
    # Signal magnitude area (for 3-axis signals)
    if window.shape[1] >= 3:
        sma = np.mean(np.sum(np.abs(window[:, :3]), axis=1))
    else:
        sma = np.mean(np.sum(np.abs(window), axis=1))
    
    # Combine features
    features = np.concatenate([mean_val, std_val, rms_val, range_val, [sma]])
    return features

In [57]:
def window_and_extract_features(emg_dict, imu_dict, fs_emg=963, fs_imu=148.148,
                                 window_sec=0.20, overlap_sec=0.10):
    """
    Window EMG and IMU data separately (accounting for different sampling rates)
    and extract features from aligned windows.
    """
    # Calculate window parameters
    emg_win_size = int(window_sec * fs_emg)
    emg_step = int((window_sec - overlap_sec) * fs_emg)
    
    imu_win_size = int(window_sec * fs_imu)
    imu_step = int((window_sec - overlap_sec) * fs_imu)
    
    all_features = []
    
    # Sort sensors for consistent ordering
    emg_sensors = sorted(emg_dict.keys())
    imu_sensors = sorted(imu_dict.keys())
    
    # Find minimum number of windows across all sensors
    min_windows = float('inf')
    
    for sensor in emg_sensors:
        n_windows = (len(emg_dict[sensor]) - emg_win_size) // emg_step
        min_windows = min(min_windows, n_windows)
    
    for sensor in imu_sensors:
        n_windows = (len(imu_dict[sensor]) - imu_win_size) // imu_step
        min_windows = min(min_windows, n_windows)
    
    # Extract features window by window
    for win_idx in range(max(1, min_windows)):
        window_features = []
        
        # EMG features
        for sensor in emg_sensors:
            start = win_idx * emg_step
            end = start + emg_win_size
            
            if end > len(emg_dict[sensor]):
                break
                
            window = emg_dict[sensor][start:end].flatten()
            
            feats = extract_emg_features(window)
            window_features.extend(feats)
        
        # IMU features
        for sensor in imu_sensors:
            start = win_idx * imu_step
            end = start + imu_win_size
            
            if end > len(imu_dict[sensor]):
                break
                
            window = imu_dict[sensor][start:end]
            
            feats = extract_imu_features(window)
            window_features.extend(feats)
        
        if len(window_features) > 0:
            all_features.append(window_features)
    
    return np.array(all_features)

In [58]:
def process_trial(csv_path, label):
    """Load and process a single trial"""
    emg_dict, imu_dict, time_dict, fs_emg, fs_imu = load_emg_imu(csv_path)
    
    print(f"  Loaded: {len(emg_dict)} EMG sensors, {len(imu_dict)} IMU sensors")
    
    X = window_and_extract_features(emg_dict, imu_dict, fs_emg, fs_imu)
    y = np.full(X.shape[0], label)
    
    return X, y

In [59]:
def load_all_trials(class_trials):
    """Load all trials organized by class"""
    X_all = []
    y_all = []
    
    for label, files in sorted(class_trials.items()):
        print(f"\nProcessing Class {label}: {len(files)} files")
        for file in files:
            print(f"  File: {os.path.basename(file)}")
            try:
                X_trial, y_trial = process_trial(file, label)
                print(f"    Generated {X_trial.shape[0]} windows with {X_trial.shape[1]} features")
                X_all.append(X_trial)
                y_all.append(y_trial)
            except Exception as e:
                print(f"    ERROR: {e}")
                continue
    
    if len(X_all) == 0:
        raise ValueError("No data was successfully loaded!")
    
    X_all = np.vstack(X_all)
    y_all = np.concatenate(y_all)
    
    return X_all, y_all

In [60]:
def prepare_data(X, y, test_size=0.2, random_state=42):
    """Split and scale data"""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=random_state
    )
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    return X_train, X_test, y_train, y_test, scaler

In [61]:
def train_evaluate(X_train, X_test, y_train, y_test, use_cv=True):
    """Train and evaluate LDA classifier"""
    clf = LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')
    
    if use_cv and len(np.unique(y_train)) > 1:
        cv_scores = cross_val_score(clf, X_train, y_train, cv=min(5, len(y_train)//2))
        print(f"\nCross-validation scores: {cv_scores}")
        print(f"Mean CV accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    
    clf.fit(X_train, y_train)
    
    y_pred = clf.predict(X_test)
    test_acc = accuracy_score(y_test, y_pred)
    
    print(f"\nTest Accuracy: {test_acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    
    return clf

In [62]:
def get_trials_from_folder(data_dir):
    """
    Scan folder and parse filenames to assign class labels.
    Expects format: "0.1_name.csv", "1.2_name.csv", etc.
    """
    trial_files = glob.glob(os.path.join(data_dir, "*.csv"))
    class_trials = {}
    
    for f in trial_files:
        basename = os.path.basename(f)
        try:
            class_label = int(basename.split(".")[0])
        except (ValueError, IndexError):
            print(f"  Skipping file with unexpected format: {basename}")
            continue
        
        class_trials.setdefault(class_label, []).append(f)
    
    return class_trials

In [63]:
def get_trials_from_multiple_folders(data_dirs):
    """
    Scan multiple folders and combine all trials.
    
    Parameters:
        data_dirs: list of folder paths OR single folder path string
    
    Returns:
        class_trials: dict {class_label: [file_paths]}
        folder_info: dict with metadata about each folder
    """
    # Handle single folder or list of folders
    if isinstance(data_dirs, str):
        data_dirs = [data_dirs]
    
    combined_trials = {}
    folder_info = {}
    
    print(f"\nScanning {len(data_dirs)} folder(s) for data...")
    
    for folder in data_dirs:
        if not os.path.exists(folder):
            print(f"  ⚠ Warning: Folder not found: {folder}")
            continue
        
        print(f"\n  Folder: {folder}")
        folder_trials = get_trials_from_folder(folder)
        
        if len(folder_trials) == 0:
            print(f"    No valid CSV files found")
            continue
        
        # Track folder metadata
        folder_info[folder] = {
            'n_classes': len(folder_trials),
            'n_files': sum(len(files) for files in folder_trials.values()),
            'classes': list(folder_trials.keys())
        }
        
        # Merge into combined trials
        for label, files in folder_trials.items():
            combined_trials.setdefault(label, []).extend(files)
            print(f"    Class {label}: {len(files)} files")
    
    return combined_trials, folder_info

In [64]:
def save_model_and_scaler(clf, scaler, model_dir="models", model_name=None):
    """
    Save trained classifier and scaler to a folder with metadata.
    
    Parameters:
        clf: Trained classifier (e.g., LDA)
        scaler: Fitted StandardScaler
        model_dir: Directory to save models (default: "models")
        model_name: Custom name for this model (default: auto-generated with timestamp)
    
    Returns:
        save_path: Path to the saved model directory
    """
    from datetime import datetime
    import json
    
    # Create models directory if it doesn't exist
    os.makedirs(model_dir, exist_ok=True)
    
    # Generate model name if not provided
    if model_name is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        model_name = f"model_{timestamp}"
    
    # Create subfolder for this model
    save_path = os.path.join(model_dir, model_name)
    os.makedirs(save_path, exist_ok=True)
    
    # Save model
    model_file = os.path.join(save_path, "trained_model.pkl")
    with open(model_file, 'wb') as f:
        pickle.dump(clf, f)
    print(f"✓ Model saved to: {model_file}")
    
    # Save scaler
    scaler_file = os.path.join(save_path, "scaler.pkl")
    with open(scaler_file, 'wb') as f:
        pickle.dump(scaler, f)
    print(f"✓ Scaler saved to: {scaler_file}")
    
    # Save metadata
    metadata = {
        'timestamp': datetime.now().isoformat(),
        'model_name': model_name,
        'model_type': type(clf).__name__,
        'n_features': clf.coef_.shape[1] if hasattr(clf, 'coef_') else None,
        'n_classes': len(clf.classes_) if hasattr(clf, 'classes_') else None,
        'classes': clf.classes_.tolist() if hasattr(clf, 'classes_') else None,
        'class_names': {
            0: "Neutral",
            1: "Pinching",
            2: "Grasping",
            3: "Zipping"
        }
    }
    
    metadata_file = os.path.join(save_path, "metadata.json")
    with open(metadata_file, 'w') as f:
        json.dump(metadata, f, indent=4)
    print(f"✓ Metadata saved to: {metadata_file}")
    
    # Create a README for this model
    readme_content = f"""# Model: {model_name}

## Model Information
- **Created**: {metadata['timestamp']}
- **Model Type**: {metadata['model_type']}
- **Number of Features**: {metadata['n_features']}
- **Number of Classes**: {metadata['n_classes']}

## Classes
"""
    for class_id, class_name in metadata['class_names'].items():
        readme_content += f"- {class_id}: {class_name}\n"

    readme_content += f"""
## Files
- `trained_model.pkl`: Trained classifier
- `scaler.pkl`: Feature scaler
- `metadata.json`: Model metadata
- `README.md`: This file

## Usage in Real-Time
Load this model for real-time classification by specifying the model name.
"""
    
    readme_file = os.path.join(save_path, "README.md")
    with open(readme_file, 'w') as f:
        f.write(readme_content)
    print(f"✓ README saved to: {readme_file}")
    
    print(f"\n{'='*60}")
    print(f"MODEL SAVED SUCCESSFULLY")
    print(f"{'='*60}")
    print(f"Location: {save_path}")
    print(f"Use model_name='{model_name}' for real-time classification")
    
    return save_path


In [65]:
if __name__ == "__main__":
    data_dirs = [
        "20251202-Data"
    ]
    
    print("="*70)
    print("EMG + IMU CLASSIFICATION PIPELINE")
    print("="*70)
    
    # Parse files from all folders
    class_trials, folder_info = get_trials_from_multiple_folders(data_dirs)
    
    # Display summary
    print("\n" + "="*70)
    print("DATA SUMMARY")
    print("="*70)
    print(f"\nTotal folders processed: {len(folder_info)}")
    for folder, info in folder_info.items():
        print(f"\n  {folder}:")
        print(f"    Classes: {info['n_classes']}, Files: {info['n_files']}")
    
    print(f"\n{'='*70}")
    print(f"COMBINED DATA ACROSS ALL FOLDERS")
    print(f"{'='*70}")
    print(f"\nFound {len(class_trials)} unique classes:")
    for label, files in sorted(class_trials.items()):
        print(f"  Class {label}: {len(files)} files total")
    
    if len(class_trials) == 0:
        print(f"\n⚠ No valid CSV files found in any folder!")
        print("Please check:")
        print("  - Folder paths are correct")
        print("  - CSV files exist in folders")
        print("  - Filenames start with class number (e.g., '0.1_trial.csv')")
    else:
        # Load and process
        print("\n" + "="*70)
        print("LOADING AND FEATURE EXTRACTION")
        print("="*70)
        X, y = load_all_trials(class_trials)
        
        print(f"\n{'='*70}")
        print(f"DATASET STATISTICS")
        print(f"{'='*70}")
        print(f"Total samples: {X.shape[0]}")
        print(f"Features per sample: {X.shape[1]}")
        
        # Detailed class distribution
        class_counts = dict(zip(*np.unique(y, return_counts=True)))
        print(f"\nClass distribution:")
        for label in sorted(class_counts.keys()):
            count = class_counts[label]
            percentage = (count / len(y)) * 100
            print(f"  Class {label}: {count:5d} samples ({percentage:5.1f}%)")
        
        # Train and evaluate
        print(f"\n{'='*70}")
        print("TRAINING AND EVALUATION")
        print("="*70)
        X_train, X_test, y_train, y_test, scaler = prepare_data(X, y)
        print(f"Training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")
        clf = train_evaluate(X_train, X_test, y_train, y_test, use_cv=True)

        # Save for real-time use
        print(f"\n{'='*70}")
        print("SAVING MODEL")
        print("="*70)
        save_model_and_scaler(clf, scaler, model_name="model_latest")
        print("\n✓ Model ready for real-time classification!")
        

EMG + IMU CLASSIFICATION PIPELINE

Scanning 1 folder(s) for data...

  Folder: 20251202-Data
    Class 0: 3 files
    Class 1: 4 files
    Class 2: 4 files
    Class 3: 2 files

DATA SUMMARY

Total folders processed: 1

  20251202-Data:
    Classes: 4, Files: 13

COMBINED DATA ACROSS ALL FOLDERS

Found 4 unique classes:
  Class 0: 3 files total
  Class 1: 4 files total
  Class 2: 4 files total
  Class 3: 2 files total

LOADING AND FEATURE EXTRACTION

Processing Class 0: 3 files
  File: 0.1_20251202.csv
  Loaded: 12 EMG sensors, 72 IMU sensors
    Generated 321 windows with 456 features
  File: 0.2_20251202.csv
  Loaded: 12 EMG sensors, 72 IMU sensors
    Generated 313 windows with 456 features
  File: 0.3_20251202.csv
  Loaded: 12 EMG sensors, 72 IMU sensors
    Generated 650 windows with 456 features

Processing Class 1: 4 files
  File: 1.1_20251202.csv
  Loaded: 12 EMG sensors, 72 IMU sensors
    Generated 423 windows with 456 features
  File: 1.2_20251202.csv
  Loaded: 12 EMG sensor

#### Real-Time Prediction

In [66]:
def test_saved_model(model_path="trained_model.pkl", 
                     scaler_path="scaler.pkl"):
    """Test that saved model can be loaded"""
    try:
        with open(model_path, 'rb') as f:
            clf = pickle.load(f)
        
        with open(scaler_path, 'rb') as f:
            scaler = pickle.load(f)
        
        print("✓ Model loaded successfully")
        print(f"  Model type: {type(clf).__name__}")
        print(f"  Classes: {clf.classes_}")
        print(f"  Expected features: {clf.coef_.shape[1] if hasattr(clf, 'coef_') else 'Unknown'}")
        
        return True
    except Exception as e:
        print(f"✗ Error loading model: {e}")
        return False

if __name__ == "__main__":
    print("="*60)
    print(test_saved_model())
    print("="*60)

✓ Model loaded successfully
  Model type: LinearDiscriminantAnalysis
  Classes: [0 1 2 3]
  Expected features: 456
True


In [67]:
"""
Real-Time EMG+IMU Classification with Delsys Trigno System
Connects to Delsys API, streams data, and performs live gesture classification
"""

import numpy as np
import socket
import struct
import time
import pickle
from collections import deque
from threading import Thread, Lock
from scipy.signal import butter, filtfilt, iirnotch
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

In [68]:

# ============================================================
# DELSYS API CONNECTION
# ============================================================

class DelsysClient:
    """
    Client for connecting to Delsys Trigno system.
    
    Default Delsys ports:
    - Command port: 50040
    - EMG data port: 50043 (2000 Hz)
    - Auxiliary (IMU) data port: 50044 (148.148 Hz)
    """
    
    def __init__(self, host='localhost', cmd_port=50040, 
                 emg_port=50043, aux_port=50044):
        self.host = host
        self.cmd_port = cmd_port
        self.emg_port = emg_port
        self.aux_port = aux_port
        
        self.cmd_socket = None
        self.emg_socket = None
        self.aux_socket = None
        
        self.is_streaming = False
        
    def connect(self):
        """Establish connection to Delsys system"""
        try:
            # Command socket
            self.cmd_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            self.cmd_socket.connect((self.host, self.cmd_port))
            print(f"✓ Connected to command port: {self.host}:{self.cmd_port}")
            
            # EMG data socket
            self.emg_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            self.emg_socket.connect((self.host, self.emg_port))
            print(f"✓ Connected to EMG port: {self.host}:{self.emg_port}")
            
            # Auxiliary (IMU) data socket
            self.aux_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            self.aux_socket.connect((self.host, self.aux_port))
            print(f"✓ Connected to AUX port: {self.host}:{self.aux_port}")
            
            return True
            
        except Exception as e:
            print(f"✗ Connection failed: {e}")
            return False
    
    def send_command(self, command):
        """Send command to Delsys system"""
        if self.cmd_socket:
            self.cmd_socket.send(f"{command}\r\n\r\n".encode())
            time.sleep(0.1)
            
    def start_streaming(self):
        """Start data streaming"""
        self.send_command("START")
        self.is_streaming = True
        print("✓ Started streaming")
        
    def stop_streaming(self):
        """Stop data streaming"""
        self.send_command("STOP")
        self.is_streaming = False
        print("✓ Stopped streaming")
        
    def disconnect(self):
        """Close all connections"""
        if self.is_streaming:
            self.stop_streaming()
        
        if self.cmd_socket:
            self.cmd_socket.close()
        if self.emg_socket:
            self.emg_socket.close()
        if self.aux_socket:
            self.aux_socket.close()
        
        print("✓ Disconnected")

In [ ]:
# ============================================================
# REAL-TIME DATA PROCESSOR
# ============================================================

class RealtimeProcessor:
    """
    Processes streaming EMG and IMU data in real-time.
    Manages circular buffers, windowing, and feature extraction.
    Uses the SAME feature extraction functions as training.
    """
    
    def __init__(self, n_emg_channels=4, n_imu_channels=6,
                 fs_emg=963, fs_imu=148.148,
                 window_sec=0.20, overlap_sec=0.10):
        
        self.n_emg_channels = n_emg_channels
        self.n_imu_channels = n_imu_channels
        self.fs_emg = fs_emg
        self.fs_imu = fs_imu
        
        self.window_sec = window_sec
        self.overlap_sec = overlap_sec
        
        # Calculate buffer sizes
        self.emg_win_size = int(window_sec * fs_emg)
        self.imu_win_size = int(window_sec * fs_imu)
        
        self.emg_step = int((window_sec - overlap_sec) * fs_emg)
        self.imu_step = int((window_sec - overlap_sec) * fs_imu)
        
        # Circular buffers for streaming data
        self.emg_buffer = deque(maxlen=self.emg_win_size * 2)
        self.imu_buffer = deque(maxlen=self.imu_win_size * 2)
        
        self.emg_sample_count = 0
        self.imu_sample_count = 0
        
        self.lock = Lock()
    
    def add_emg_sample(self, sample):
        """Add EMG sample to buffer (sample should be array of n_channels)"""
        with self.lock:
            self.emg_buffer.append(sample)
            self.emg_sample_count += 1
    
    def add_imu_sample(self, sample):
        """Add IMU sample to buffer (sample should be array of n_channels)"""
        with self.lock:
            self.imu_buffer.append(sample)
            self.imu_sample_count += 1
    
    def is_window_ready(self):
        """Check if we have enough data for a new window"""
        emg_ready = len(self.emg_buffer) >= self.emg_win_size
        imu_ready = len(self.imu_buffer) >= self.imu_win_size
        
        emg_step_ready = self.emg_sample_count >= self.emg_step
        imu_step_ready = self.imu_sample_count >= self.imu_step
        
        return emg_ready and imu_ready and emg_step_ready and imu_step_ready
    
    def extract_window_features(self):
        """
        Extract features from current window.
        Uses the SAME preprocessing and feature extraction as training.
        """
        with self.lock:
            if not self.is_window_ready():
                return None
            
            # Get windows
            emg_window = np.array(list(self.emg_buffer)[-self.emg_win_size:])
            imu_window = np.array(list(self.imu_buffer)[-self.imu_win_size:])
            
            # Reset step counters
            self.emg_sample_count = 0
            self.imu_sample_count = 0
        
        # Extract features
        features = []
        
        # EMG features - process each channel
        for ch in range(self.n_emg_channels):
            signal = emg_window[:, ch]
            
            # Extract features using training function
            feats = extract_emg_features(signal, fs=self.fs_emg)
            features.extend(feats)
        
        # Extract IMU features using training function
        imu_feats = extract_imu_features(imu_window)
        features.extend(imu_feats)
        
        return np.array(features)


In [70]:
# ============================================================
# REAL-TIME CLASSIFIER
# ============================================================

class RealtimeClassifier:
    """
    Real-time gesture classifier.
    Loads trained model and scaler, performs predictions on streaming data.
    """
    
    def __init__(self, model_path, scaler_path, class_names=None):
        # Load trained model and scaler
        with open(model_path, 'rb') as f:
            self.model = pickle.load(f)
        
        with open(scaler_path, 'rb') as f:
            self.scaler = pickle.load(f)
        
        self.class_names = class_names or {
            0: "Neutral",
            1: "Pinching", 
            2: "Grasping",
            3: "Zipping"
        }
        
        # Prediction smoothing
        self.prediction_buffer = deque(maxlen=5)
        
    def predict(self, features):
        """Predict class from features"""
        if features is None:
            return None, None
        
        # Scale features
        features_scaled = self.scaler.transform(features.reshape(1, -1))
        
        # Predict
        pred_label = self.model.predict(features_scaled)[0]
        pred_proba = self.model.predict_proba(features_scaled)[0]
        
        return pred_label, pred_proba
    
    def predict_smoothed(self, features):
        """Predict with temporal smoothing"""
        pred_label, pred_proba = self.predict(features)
        
        if pred_label is not None:
            self.prediction_buffer.append(pred_label)
            
            # Majority vote
            if len(self.prediction_buffer) > 0:
                smoothed_pred = max(set(self.prediction_buffer), 
                                   key=self.prediction_buffer.count)
                return smoothed_pred, pred_proba
        
        return pred_label, pred_proba
    
    def get_class_name(self, label):
        """Get class name from label"""
        return self.class_names.get(label, f"Unknown ({label})")

In [71]:
# ============================================================
# MAIN REAL-TIME SYSTEM
# ============================================================

class RealtimeEMGSystem:
    """
    Complete real-time EMG+IMU classification system.
    Integrates Delsys client, data processor, and classifier.
    """
    
    def __init__(self, model_path, scaler_path, 
                 n_emg_channels=4, n_imu_channels=6,
                 host='localhost'):
        
        self.client = DelsysClient(host=host)
        self.processor = RealtimeProcessor(
            n_emg_channels=n_emg_channels,
            n_imu_channels=n_imu_channels
        )
        self.classifier = RealtimeClassifier(model_path, scaler_path)
        
        self.is_running = False
        self.emg_thread = None
        self.imu_thread = None
        
    def connect(self):
        """Connect to Delsys system"""
        return self.client.connect()
    
    def _stream_emg_data(self):
        """Thread function to stream EMG data"""
        bytes_per_sample = 4  # float32
        bytes_per_channel = bytes_per_sample * self.processor.n_emg_channels
        
        while self.is_running:
            try:
                data = self.client.emg_socket.recv(bytes_per_channel)
                if len(data) == bytes_per_channel:
                    # Unpack float values
                    values = struct.unpack('f' * self.processor.n_emg_channels, data)
                    self.processor.add_emg_sample(np.array(values))
            except Exception as e:
                if self.is_running:
                    print(f"EMG streaming error: {e}")
                break
    
    def _stream_imu_data(self):
        """Thread function to stream IMU data"""
        bytes_per_sample = 4  # float32
        bytes_per_channel = bytes_per_sample * self.processor.n_imu_channels
        
        while self.is_running:
            try:
                data = self.client.aux_socket.recv(bytes_per_channel)
                if len(data) == bytes_per_channel:
                    values = struct.unpack('f' * self.processor.n_imu_channels, data)
                    self.processor.add_imu_sample(np.array(values))
            except Exception as e:
                if self.is_running:
                    print(f"IMU streaming error: {e}")
                break
    
    def start(self):
        """Start real-time classification"""
        if not self.client.is_streaming:
            self.client.start_streaming()
        
        self.is_running = True
        
        # Start streaming threads
        self.emg_thread = Thread(target=self._stream_emg_data, daemon=True)
        self.imu_thread = Thread(target=self._stream_imu_data, daemon=True)
        
        self.emg_thread.start()
        self.imu_thread.start()
        
        print("\n" + "="*60)
        print("REAL-TIME CLASSIFICATION STARTED")
        print("="*60)
        print("Press Ctrl+C to stop\n")
        
        try:
            while self.is_running:
                if self.processor.is_window_ready():
                    features = self.processor.extract_window_features()
                    
                    if features is not None:
                        pred_label, pred_proba = self.classifier.predict_smoothed(features)
                        class_name = self.classifier.get_class_name(pred_label)
                        confidence = pred_proba[pred_label] * 100
                        
                        # Display prediction
                        print(f"\r{class_name:12s} | Confidence: {confidence:5.1f}%", 
                              end='', flush=True)
                
                time.sleep(0.01)  # Small delay to prevent CPU overload
                
        except KeyboardInterrupt:
            print("\n\nStopping...")
            self.stop()
    
    def stop(self):
        """Stop real-time classification"""
        self.is_running = False
        
        if self.emg_thread:
            self.emg_thread.join(timeout=1)
        if self.imu_thread:
            self.imu_thread.join(timeout=1)
        
        self.client.disconnect()
        print("✓ System stopped")

In [73]:
# ============================================================
# USAGE EXAMPLE
# ============================================================

if __name__ == "__main__":
    # Configuration
    MODEL_NAME = "model_latest"  # Name of saved model folder
    MODEL_DIR = "models"  # Directory containing saved models
    
    # Construct paths
    MODEL_PATH = os.path.join(MODEL_DIR, MODEL_NAME, "trained_model.pkl")
    SCALER_PATH = os.path.join(MODEL_DIR, MODEL_NAME, "scaler.pkl")
    
    HOST = 'localhost'  # Change to Delsys computer IP if remote
    N_EMG_CHANNELS = 4  # Number of EMG sensors
    N_IMU_CHANNELS = 6  # 6-axis IMU (3 acc + 3 gyro)
    
    # Verify model exists
    if not os.path.exists(MODEL_PATH):
        print(f"Error: Model not found at {MODEL_PATH}")
        print(f"\nAvailable models in '{MODEL_DIR}':")
        if os.path.exists(MODEL_DIR):
            for item in os.listdir(MODEL_DIR):
                print(f"  - {item}")
        else:
            print(f"  Models directory '{MODEL_DIR}' does not exist")
        exit(1)
    
    # Create system
    print("="*70)
    print("REAL-TIME EMG+IMU CLASSIFICATION")
    print("="*70)
    print(f"\nLoading model: {MODEL_NAME}")
    
    system = RealtimeEMGSystem(
        model_path=MODEL_PATH,
        scaler_path=SCALER_PATH,
        n_emg_channels=N_EMG_CHANNELS,
        n_imu_channels=N_IMU_CHANNELS,
        host=HOST
    )
    
    # Connect and start
    print("\nConnecting to Delsys system...")
    if system.connect():
        print("\nStarting real-time classification...")
        system.start()
    else:
        print("Failed to connect to Delsys system")
        print("\nTroubleshooting:")
        print("1. Ensure Trigno Control Utility is running")
        print("2. Check host IP address")
        print("3. Verify firewall settings")
        print("4. Check that sensors are paired and active")

REAL-TIME EMG+IMU CLASSIFICATION

Loading model: model_latest

Connecting to Delsys system...
✗ Connection failed: [WinError 10061] No connection could be made because the target machine actively refused it
Failed to connect to Delsys system

Troubleshooting:
1. Ensure Trigno Control Utility is running
2. Check host IP address
3. Verify firewall settings
4. Check that sensors are paired and active
